# BM25 Retrieval

This notebook implements a BM25-based retrieval system for the course material from **Verifikacija softvera**.

The retrieval system indexes preprocessed text chunks and retrieves the most relevant chunks for a given question. Retrieval performance is evaluated using MRR, Hit@K, Precision@K, Recall@K, and nDCG@K.


In [24]:
%pip install -q rank-bm25

import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

from rank_bm25 import BM25Okapi

Note: you may need to restart the kernel to use updated packages.


In [25]:
TOP_K_VALUES = (1, 3, 5, 10)
PROJECT_ROOT = Path.cwd()

CHUNKS_PREPROCESSED_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks_preprocessed.jsonl"
)

TRAIN_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "train.jsonl"
)

VALIDATION_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "validation.jsonl"
)

TEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "test.jsonl"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "retrieval"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert CHUNKS_PREPROCESSED_PATH.exists()
assert TRAIN_PATH.exists()
assert VALIDATION_PATH.exists()
assert TEST_PATH.exists()

print("Chunkovi:", CHUNKS_PREPROCESSED_PATH)
print("Trening skup:", TRAIN_PATH)
print("Validacioni skup:", VALIDATION_PATH)
print("Test skup:", TEST_PATH)

Chunkovi: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/processed/chunks_preprocessed.jsonl
Trening skup: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/splits/train.jsonl
Validacioni skup: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/splits/validation.jsonl
Test skup: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/splits/test.jsonl


In [26]:
def load_jsonl(path: Path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Neispravan JSON u redu {line_number}: {path}"
                ) from error

    return records


print("Funkcija load_jsonl je definisana.")

Funkcija load_jsonl je definisana.


In [27]:
chunks = load_jsonl(CHUNKS_PREPROCESSED_PATH)

train_data = load_jsonl(TRAIN_PATH)
validation_data = load_jsonl(VALIDATION_PATH)
test_data = load_jsonl(TEST_PATH)

print(f"Broj chunkova: {len(chunks)}")
print(f"Broj pitanja u trening skupu: {len(train_data)}")
print(f"Broj pitanja u validacionom skupu: {len(validation_data)}")
print(f"Broj pitanja u test skupu: {len(test_data)}")

Broj chunkova: 356
Broj pitanja u trening skupu: 100
Broj pitanja u validacionom skupu: 21
Broj pitanja u test skupu: 22


In [28]:
assert chunks

assert all(
    isinstance(chunk.get("processed_text"), str)
    and chunk["processed_text"].strip()
    for chunk in chunks
)

for split_name, data in {
    "train": train_data,
    "validation": validation_data,
    "test": test_data
}.items():

    assert data, f"{split_name} split is empty"

    assert all(
        isinstance(example.get("processed_question"), str)
        and example["processed_question"].strip()
        for example in data
    ), f"Missing processed_question in {split_name}"

required_chunk_fields = {
    "chunk_id",
    "pdf_page_start",
    "pdf_page_end"
}

assert required_chunk_fields.issubset(chunks[0]), "Nedostaju metadata polja chunk-a."

assert len(
    {chunk["chunk_id"] for chunk in chunks}
) == len(chunks), "chunk_id mora biti jedinstven."

In [29]:
chunks_df = pd.DataFrame(chunks)

train_df = pd.DataFrame(train_data)
validation_df = pd.DataFrame(validation_data)
test_df = pd.DataFrame(test_data)

chunks_df[
    [
        "chunk_id",
        "pdf_page_start",
        "pdf_page_end",
        "processed_text"
    ]
].head(3)

,chunk_id,pdf_page_start,pdf_page_end,processed_text
0,chunk_0001,17,17,Pregled 1.1 Upravljanje kvalitetom softvera . ...
1,chunk_0002,17,18,"nu ulogu u razvoju veštačke inteligencije, obr..."
2,chunk_0003,17,18,koriste kako bi se na vreme zadovoljili korisn...


In [30]:
chunk_ids = chunks_df["chunk_id"].tolist()

chunk_texts = chunks_df["processed_text"].tolist()

print(f"Broj chunkova: {len(chunk_texts)}")
print(f"Broj ID-ova chunkova: {len(chunk_ids)}")

Broj chunkova: 356
Broj ID-ova chunkova: 356


In [31]:
def normalize_for_bm25(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    text = text.replace("�", "")
    text = re.sub(r"(?<=\w)-\s*(?=\w)", "", text)
    return text.lower()


TOKEN_PATTERN = re.compile(r"[\w]+", re.UNICODE)


def tokenize(text: str):
    return TOKEN_PATTERN.findall(
        normalize_for_bm25(text)
    )


tokenized_documents = [
    tokenize(text)
    for text in chunk_texts
]

assert all(
    tokens
    for tokens in tokenized_documents
), "Korpus sadrži chunk koji se tokenizuje u praznu listu."

In [32]:
bm25_index = BM25Okapi(tokenized_documents)

avg_tokens_per_chunk = (
    sum(map(len, tokenized_documents)) / len(tokenized_documents)
)

print("BM25 indeks je izgrađen za:", len(tokenized_documents), "chunkova")
print(f"Prosečan broj tokena po chunk-u: {avg_tokens_per_chunk:.1f}")
print(f"Veličina rečnika: {len(bm25_index.idf)}")

BM25 indeks je izgrađen za: 356 chunkova
Prosečan broj tokena po chunk-u: 150.1
Veličina rečnika: 8014


In [33]:
from collections import Counter

chunk_index = 0

document_tokens = tokenized_documents[chunk_index]

term_frequencies = Counter(document_tokens)

term_weights = {
    term: freq * bm25_index.idf.get(term, 0.0)
    for term, freq in term_frequencies.items()
}

top_terms = sorted(
    term_weights.items(),
    key=lambda item: item[1],
    reverse=True
)[:15]

bm25_terms_df = pd.DataFrame(
    top_terms,
    columns=["term", "bm25_weight"]
)

bm25_terms_df

,term,bm25_weight
0,industrija,10.936120
1,koji,9.607049
2,za,9.607049
3,i,9.607049
4,važni,7.683201
5,standardi,7.198946
6,razvija,6.324246
7,atributi,5.495138
8,najdinamičnijih,5.468060
9,svetu,5.468060


In [34]:
example = validation_data[0]

query = example["processed_question"]

query_tokens = tokenize(query)

query_weights = {
    term: bm25_index.idf.get(term, 0.0)
    for term in set(query_tokens)
    if term in bm25_index.idf
}

query_terms_df = pd.DataFrame(
    query_weights.items(),
    columns=["term", "idf"]
).sort_values(
    "idf",
    ascending=False
)

query_terms_df

,term,idf
0,služi,5.468060
2,cachegrind,3.987559
4,šta,3.402646
5,se,1.200881
6,je,1.200881
7,i,1.200881
8,za,1.200881
1,koristi,1.186455
3,kako,1.065276


In [35]:
def retrieve_chunks(
    lexical_question: str,
    top_k: int = 5
) -> pd.DataFrame:

    question_tokens = tokenize(lexical_question)

    if not question_tokens:
        return pd.DataFrame(columns=[
            "rank",
            "chunk_id",
            "score",
            "pdf_page_start",
            "pdf_page_end",
            "processed_text",    # bolja čitljivost
        ])

    scores = bm25_index.get_scores(question_tokens)

    top_indices = np.argsort(
        scores
    )[::-1][:top_k]

    results = []

    for rank, index in enumerate(top_indices, start=1):
        chunk = chunks[index]

        results.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "score": float(scores[index]),
            "pdf_page_start": chunk["pdf_page_start"],
            "pdf_page_end": chunk["pdf_page_end"],
            "processed_text": chunk["processed_text"]   # bolja čitljivost
        })

    return pd.DataFrame(results)

In [36]:
def get_relevant_chunk_ids(gold_pages):
    if isinstance(gold_pages, int):
        gold_pages = [gold_pages]

    gold_pages = set(gold_pages)
    relevant_ids = set()

    for chunk in chunks:
        chunk_pages = set(
            range(
                int(chunk["pdf_page_start"]),
                int(chunk["pdf_page_end"]) + 1
            )
        )

        if chunk_pages & gold_pages:
            relevant_ids.add(chunk["chunk_id"])

    return relevant_ids

In [37]:
example = validation_data[1]

print("Original question:")
print(example["question"])

print("\nProcessed question:")
print(example["processed_question"])

print("\nGold source pages:")
print(example["source_pages"])

Original question:
Šta je instrumentaciono profajliranje?

Processed question:
Šta je instrumentaciono profajliranje?

Gold source pages:
[188, 189]


In [38]:
def overlaps_gold_pages(
    row,
    source_pages
):
    return any(
        row["pdf_page_start"] <= page <= row["pdf_page_end"]
        for page in source_pages
    )

In [39]:
retrieved = retrieve_chunks(
    example["processed_question"],
    top_k=10
)

retrieved["relevant"] = retrieved.apply(
    lambda row: overlaps_gold_pages(
        row,
        example["source_pages"]
    ),
    axis=1
)

retrieved[
    [
        "rank",
        "score",
        "pdf_page_start",
        "pdf_page_end",
        "relevant",
        "processed_text"
    ]
]

,rank,score,pdf_page_start,pdf_page_end,relevant,processed_text
0,1,10.105061,185,185,False,[Profajleri] ogućava programu da se izvršava g...
1,2,9.222286,188,188,True,[Profajliranje i dinamičko detektovanje grešak...
2,3,8.723421,174,175,False,[Profajliranje i dinamičko detektovanje grešak...
3,4,7.120880,172,173,False,"ovanja, tačaka prekida i tačaka posmatranja. P..."
4,5,5.851407,188,189,True,[Profajliranje i dinamičko detektovanje grešak...
5,6,5.509786,169,170,False,[Štampanje umesto debagera] debagera i trebalo...
6,7,5.347021,189,189,True,[Profajleri] i prati ponašanje programa u real...
7,8,5.312696,153,154,False,"[Vrste debagovanja] ive u memoriji, tj. da li ..."
8,9,5.161910,204,204,False,[Profajliranje i dinamičko detektovanje grešak...
9,10,5.160933,192,192,False,[Profajliranje i dinamičko detektovanje grešak...


In [40]:
example = validation_data[0]

question_scores = bm25_index.get_scores(
    tokenize(example["processed_question"])
)

score_df = pd.DataFrame({
    "chunk_index": range(len(question_scores)),
    "score": question_scores
}).sort_values(
    "score",
    ascending=False
)

top_score_df = score_df.head(20)

top_score_df

,chunk_index,score
296,296,22.075323
293,293,21.199117
289,289,16.848863
286,286,16.396171
135,135,15.537117
106,106,15.434723
0,0,15.258765
87,87,14.313371
261,261,14.277571
88,88,13.975917


In [41]:
def calculate_metrics_at_k(
    relevances,
    n_relevant,
    k
):
    rel = np.array(relevances[:k], dtype=int)

    n_retrieved_relevant = int(rel.sum())

    # Hit@k - da li smo našli makar jedan relevantan chunk
    hit = int(n_retrieved_relevant > 0)

    # Precision@k - koliko od top-k je relevantno
    precision = n_retrieved_relevant / k

    # Recall@k - koliko relevantnih chunkova smo pokrili
    recall = (
        n_retrieved_relevant / n_relevant
        if n_relevant > 0
        else 0.0
    )

    # F1@k - harmonijska sredina precisiona i recall-a
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    # MRR@k - koliko visoko se pojavio prvi relevantan chunk
    reciprocal_rank = 0.0

    for rank, relevant in enumerate(rel, start=1):
        if relevant:
            reciprocal_rank = 1.0 / rank
            break

    # nDCG@k - kvalitet celog redosleda relevantnih chunkova
    dcg = sum(
        relevant / np.log2(rank + 1)
        for rank, relevant in enumerate(rel, start=1)
    )

    ideal_relevant = min(n_relevant, k)

    idcg = sum(
        1.0 / np.log2(rank + 1)
        for rank in range(1, ideal_relevant + 1)
    )

    ndcg = dcg / idcg if idcg > 0 else 0.0

    return {
        "Hit": hit,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "MRR": reciprocal_rank,
        "nDCG": ndcg,
    }

In [42]:
def evaluate_retriever(
    data,
    top_k_values=TOP_K_VALUES
):
    max_k = max(top_k_values)

    per_query_results = []
    skipped_questions = []

    for example in data:

        relevant_ids = get_relevant_chunk_ids(
            example["source_pages"]
        )


        if not relevant_ids:
            skipped_questions.append(example["id"])
            continue

        retrieved = retrieve_chunks(
            example["processed_question"],
            top_k=max_k
        )

        retrieved_ids = retrieved["chunk_id"].tolist()

        relevances = [
            1 if chunk_id in relevant_ids else 0
            for chunk_id in retrieved_ids
        ]

        for k in top_k_values:

            metrics = calculate_metrics_at_k(
                relevances=relevances,
                n_relevant=len(relevant_ids),
                k=k
            )

            per_query_results.append({
                "question_id": example["id"],
                "k": k,
                **metrics
            })

    per_query_df = pd.DataFrame(per_query_results)

    metrics_df = (
        per_query_df
        .groupby("k")[
            [
                "Hit",
                "Precision",
                "Recall",
                "F1",
                "MRR",
                "nDCG"
            ]
        ]
        .mean()
        .reset_index()
    )

    print(f"Ukupno pitanja: {len(data)}")
    print(f"Evaluabilno: {len(data) - len(skipped_questions)}")
    print(f"Preskočeno: {len(skipped_questions)}")

    if skipped_questions:
        print("Preskočeni question IDs:", skipped_questions)

    return metrics_df, per_query_df

In [43]:
metrics_df, per_query_metrics_df = evaluate_retriever(
    validation_data
)

metrics_df

Ukupno pitanja: 21
Evaluabilno: 21
Preskočeno: 0


,k,Hit,Precision,Recall,F1,MRR,nDCG
0,1,0.142857,0.142857,0.031349,0.050265,0.142857,0.142857
1,3,0.571429,0.253968,0.176984,0.203824,0.309524,0.223996
2,5,0.619048,0.257143,0.269161,0.257082,0.321429,0.255072
3,10,0.809524,0.223810,0.460544,0.294407,0.343386,0.345595


In [44]:
import json

import joblib

ARTIFACTS_DIR = RESULTS_DIR / "bm25"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    bm25_index,
    ARTIFACTS_DIR / "bm25_index.joblib"
)

with (ARTIFACTS_DIR / "bm25_tokenized_documents.json").open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(tokenized_documents, file, ensure_ascii=False)

chunks_export = chunks_df[
    [
        "chunk_id",
        "text",
        "processed_text",
        "pdf_page_start",
        "pdf_page_end",
        "printed_page_start",
        "printed_page_end",
        "section_ref",
        "heading",
    ]
]

chunks_export.to_json(
    ARTIFACTS_DIR / "bm25_chunks.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

metadata = {
    "indexer_input": "processed_text",
    "retrieval_question_input": "processed_question",
    "generator_context_input": "processed_text",
    "top_k": max(TOP_K_VALUES),
    "n_chunks": len(chunks),
    "bm25_k1": bm25_index.k1,
    "bm25_b": bm25_index.b,
}

with (ARTIFACTS_DIR / "bm25_metadata.json").open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

metrics_df.to_csv(
    ARTIFACTS_DIR / "validation_metrics.csv",
    index=False
)

per_query_metrics_df.to_csv(
    ARTIFACTS_DIR / "validation_per_query_metrics.csv",
    index=False
)

print("BM25 artefakti su sačuvani u:", ARTIFACTS_DIR)

BM25 artefakti su sačuvani u: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/retrieval/bm25


In [45]:
def export_retrieval_results(data, split_name, top_k=10):
    export_path = RESULTS_DIR / f"bm25_{split_name}_top{top_k}.jsonl"

    with export_path.open("w", encoding="utf-8") as file:
        for example in data:
            retrieved = retrieve_chunks(
                example["processed_question"],
                top_k=top_k
            )

            record = {
                "question_id": example["id"],
                "question": example["question"],
                "processed_question": example["processed_question"],
                "answer": example["answer"],
                "source_pages": example["source_pages"],
                "retrieved_chunks": retrieved.to_dict(orient="records"),
            }

            file.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    print(f"Sačuvani retrieval rezultati za {split_name}:", export_path)


for split_name, data in {
    "train": train_data,
    "validation": validation_data,
    "test": test_data,
}.items():
    export_retrieval_results(
        data,
        split_name,
        top_k=max(TOP_K_VALUES)
    )

Sačuvani retrieval rezultati za train: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/retrieval/bm25_train_top10.jsonl
Sačuvani retrieval rezultati za validation: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/retrieval/bm25_validation_top10.jsonl
Sačuvani retrieval rezultati za test: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/retrieval/bm25_test_top10.jsonl
